# 🎭 MuseTalk: Real-Time AI Talking Avatar Generator (Google Colab Fixed)
### Generate realistic talking head avatars from a **Single Photo/Video** and **Audio File**

> ⚠️ **GPU Required:** Ensure GPU runtime is active: `Runtime` → `Change runtime type` → Select **T4 GPU** → `Save`.

---
### ✅ Why this notebook doesn't get stuck:
1. **Isolated Python 3.10**: Colab defaults to Python 3.12/3.13 which breaks MMCV. We use an isolated Python 3.10 via `micromamba`.
2. **Pre-built MMCV Wheels**: Installs pre-compiled `mmcv==2.1.0` binary wheels matching PyTorch 2.1.2 + CUDA 12.1. **Zero compiling from source, takes 15 seconds instead of 40 minutes!**
3. **NumPy 1.x ABI Locked**: Locks `numpy==1.26.4` to prevent the OpenCV/MMCV `_ARRAY_API not found` crash.
4. **No Broken Dependencies**: HuggingFace Hub, Transformers, and Gradio versions are strictly pinned to compatible releases.

In [ ]:
#@title 🚀 Step 1: Automated Setup (Python 3.10 + Dependencies + Weights)
#@markdown Run this cell once. It prepares the isolated environment, installs pre-compiled wheels, and downloads all weights (~4-5 mins).

import os, sys, shutil, subprocess, urllib.request

print("=" * 60)
print("[1/5] Verifying GPU...")
print("=" * 60)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

print("\n" + "=" * 60)
print("[2/5] Creating Isolated Python 3.10 Environment (micromamba)...")
print("=" * 60)
if not os.path.exists('/content/bin/micromamba'):
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content/ bin/micromamba > /dev/null 2>&1

if not os.path.exists('/content/env'):
    !/content/bin/micromamba create -y -p /content/env python=3.10 pip git ffmpeg -c conda-forge > /dev/null 2>&1

ENV_PYTHON = '/content/env/bin/python'
ENV_PIP = '/content/env/bin/pip'
!{ENV_PYTHON} --version

print("\n" + "=" * 60)
print("[3/5] Cloning MuseTalk Repository...")
print("=" * 60)
MUSETALK_DIR = '/content/MuseTalk'
if not os.path.exists(MUSETALK_DIR):
    !git clone -b main https://github.com/TMElyralab/MuseTalk.git {MUSETALK_DIR}
else:
    print("Repository already exists.")

%cd {MUSETALK_DIR}

print("\n" + "=" * 60)
print("[4/5] Installing PyTorch & Prebuilt OpenMMLab Wheels...")
print("=" * 60)
# 1. PyTorch 2.1.2 with CUDA 12.1
!{ENV_PIP} install -q torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121

# 2. OpenMMLab Stack (Prebuilt binary wheel for mmcv 2.1.0 on cu121/torch2.1 - NO source compilation!)
!{ENV_PIP} install -q mmengine
!{ENV_PIP} install -q mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html
!{ENV_PIP} install -q --no-build-isolation chumpy
!{ENV_PIP} install -q 'mmdet>=3.2.0' mmpose==1.1.0

# 3. Patch mmdet version guard
import glob
mmdet_inits = glob.glob('/content/env/lib/python3.10/site-packages/mmdet/__init__.py')
if mmdet_inits:
    !sed -i "s/mmcv_maximum_version = .*/mmcv_maximum_version = '2.2.0'/" {mmdet_inits[0]}

# 4. Install ML & MuseTalk dependencies with strict version pinning
!{ENV_PIP} install -q \
    diffusers==0.30.2 \
    accelerate==0.28.0 \
    soundfile==0.12.1 \
    librosa==0.11.0 \
    einops==0.8.1 \
    omegaconf \
    imageio \
    imageio-ffmpeg \
    ffmpeg-python \
    moviepy==1.0.3 \
    gdown \
    tqdm \
    pyyaml \
    matplotlib-inline \
    'gradio==4.44.1' \
    'transformers>=4.39.2,<4.45.0' \
    'huggingface_hub>=0.23.2,<1.0'

# 5. LOCK NumPy 1.x ABI & Setuptools (CRITICAL: prevents _ARRAY_API not found)
!{ENV_PIP} install -q 'numpy==1.26.4' 'opencv-python==4.9.0.80' 'setuptools<81'

print("\n" + "=" * 60)
print("[5/5] Downloading Model Weights...")
print("=" * 60)
os.makedirs('models/dwpose', exist_ok=True)
os.makedirs('models/sd-vae', exist_ok=True)
os.makedirs('models/sd-vae-ft-mse', exist_ok=True)
os.makedirs('models/whisper', exist_ok=True)
os.makedirs('models/face-parse-bisent', exist_ok=True)
os.makedirs('models/musetalk', exist_ok=True)
os.makedirs('models/musetalkV15', exist_ok=True)
os.makedirs('/content/input_data', exist_ok=True)

# DWPose
if not os.path.exists('models/dwpose/dw-ll_ucoco_384.pth') or os.path.getsize('models/dwpose/dw-ll_ucoco_384.pth') < 1000:
    !wget -q --show-progress -O models/dwpose/dw-ll_ucoco_384.pth \
        'https://huggingface.co/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.pth'

# SD-VAE (ft-mse)
if not os.path.exists('models/sd-vae/config.json'):
    !wget -q -O models/sd-vae/config.json \
        'https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/config.json'
if not os.path.exists('models/sd-vae/diffusion_pytorch_model.bin') or os.path.getsize('models/sd-vae/diffusion_pytorch_model.bin') < 1000:
    !wget -q --show-progress -O models/sd-vae/diffusion_pytorch_model.bin \
        'https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.bin'
shutil.copy2('models/sd-vae/config.json', 'models/sd-vae-ft-mse/config.json')
shutil.copy2('models/sd-vae/diffusion_pytorch_model.bin', 'models/sd-vae-ft-mse/diffusion_pytorch_model.bin')

# Face Parse BiSeNet
if not os.path.exists('models/face-parse-bisent/79999_iter.pth') or os.path.getsize('models/face-parse-bisent/79999_iter.pth') < 1000:
    !wget -q --show-progress -O models/face-parse-bisent/79999_iter.pth \
        'https://huggingface.co/ManyOtherFunctions/face-parse-bisent/resolve/main/79999_iter.pth'
if not os.path.exists('models/face-parse-bisent/resnet18-5c106cde.pth') or os.path.getsize('models/face-parse-bisent/resnet18-5c106cde.pth') < 1000:
    !wget -q --show-progress -O models/face-parse-bisent/resnet18-5c106cde.pth \
        'https://download.pytorch.org/models/resnet18-5c106cde.pth'

# MuseTalk V1.0
if not os.path.exists('models/musetalk/musetalk.json'):
    !wget -q -O models/musetalk/musetalk.json \
        'https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalk/musetalk.json'
if not os.path.exists('models/musetalk/pytorch_model.bin') or os.path.getsize('models/musetalk/pytorch_model.bin') < 1000:
    !wget -q --show-progress -O models/musetalk/pytorch_model.bin \
        'https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalk/pytorch_model.bin'
shutil.copy2('models/musetalk/musetalk.json', 'models/musetalk/config.json')

# MuseTalk V1.5
if not os.path.exists('models/musetalkV15/musetalk.json'):
    !wget -q -O models/musetalkV15/musetalk.json \
        'https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalkV15/musetalk.json'
if not os.path.exists('models/musetalkV15/unet.pth') or os.path.getsize('models/musetalkV15/unet.pth') < 1000:
    !wget -q --show-progress -O models/musetalkV15/unet.pth \
        'https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalkV15/unet.pth'
shutil.copy2('models/musetalkV15/musetalk.json', 'models/musetalkV15/config.json')

# Whisper Tiny (HF format used by transformers.WhisperModel)
whisper_files = ['config.json', 'preprocessor_config.json', 'tokenizer.json',
                 'vocab.json', 'merges.txt', 'special_tokens_map.json',
                 'tokenizer_config.json', 'generation_config.json', 'model.safetensors']
for wf in whisper_files:
    dst = f'models/whisper/{wf}'
    if not os.path.exists(dst) or os.path.getsize(dst) < 10:
        !wget -q -O {dst} 'https://huggingface.co/openai/whisper-tiny/resolve/main/{wf}'

# Whisper tiny.pt (with custom User-Agent to avoid 0-byte download)
whisper_pt = 'models/whisper/tiny.pt'
if not os.path.exists(whisper_pt) or os.path.getsize(whisper_pt) < 1000:
    !curl -sL -A 'Mozilla/5.0' -o {whisper_pt} 'https://openaipublic.blob.core.windows.net/whisper/models/65147644a518d1260e3c49e477f2925e2c8f61831a6d6415a4c7f9b180e66772/tiny.pt'

print("\n" + "=" * 60)
print("HEALTH CHECK: Verifying critical imports...")
print("=" * 60)
test_code = """
import numpy as np
import cv2
import torch
import mmcv
import mmpose
import mmdet
from transformers import WhisperModel
print('  numpy      :', np.__version__)
print('  opencv     :', cv2.__version__)
print('  torch      :', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('  mmcv       :', mmcv.__version__)
print('  mmpose     :', mmpose.__version__)
print('  mmdet      :', mmdet.__version__)
print('\u2705 ALL CRITICAL IMPORTS PASSED WITHOUT CONFLICTS!')
"""
!{ENV_PYTHON} -c "{test_code}"
print("\n\U0001f389 SETUP COMPLETE! Proceed to Step 2.")


In [ ]:
#@title 📷 Step 2: Upload Photo/Video & Audio
#@markdown Upload your avatar face image (.jpg, .png) and speech audio (.wav, .mp3).

import os, shutil, subprocess
from google.colab import files
from IPython.display import display, Image, Audio

INPUT_DIR = '/content/input_data'
os.makedirs(INPUT_DIR, exist_ok=True)

# 1. Upload Avatar Image or Video
print("📸 Upload your FACE IMAGE (.png, .jpg) or VIDEO (.mp4):")
up_img = files.upload()
if up_img:
    img_filename = list(up_img.keys())[0]
    ext = os.path.splitext(img_filename)[1].lower()
    raw_visual_path = os.path.join(INPUT_DIR, f"raw_visual{ext}")
    shutil.move(img_filename, raw_visual_path)
    print(f"✅ Visual uploaded: {raw_visual_path}")

# 2. Upload Audio File
print("\n🎧 Upload your SPEECH AUDIO (.wav, .mp3, .m4a):")
up_audio = files.upload()
if up_audio:
    audio_filename = list(up_audio.keys())[0]
    raw_audio_path = os.path.join(INPUT_DIR, audio_filename)
    shutil.move(audio_filename, raw_audio_path)
    print(f"✅ Audio uploaded: {raw_audio_path}")

# 3. Standardize Audio to 16kHz mono WAV
clean_audio_path = os.path.join(INPUT_DIR, 'speech_16k.wav')
!ffmpeg -y -i "{raw_audio_path}" -ar 16000 -ac 1 "{clean_audio_path}" -loglevel error

# Probe duration
probe_cmd = ['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', clean_audio_path]
audio_duration = float(subprocess.check_output(probe_cmd).decode().strip())
print(f"  Audio Duration: {audio_duration:.2f} seconds")

# 4. Prepare 25fps Video Template
final_video_path = os.path.join(INPUT_DIR, 'input_25fps.mp4')
if ext in ['.jpg', '.jpeg', '.png', '.webp']:
    # Loop image to match audio duration at 25fps (even dimensions for H.264)
    !ffmpeg -y -loop 1 -i "{raw_visual_path}" -c:v libx264 -t {audio_duration:.2f} \
        -pix_fmt yuv420p -vf "scale=trunc(iw/2)*2:trunc(ih/2)*2" -r 25 "{final_video_path}" -loglevel error
else:
    # Resample existing video to 25fps
    !ffmpeg -y -i "{raw_visual_path}" -r 25 -c:v libx264 -pix_fmt yuv420p "{final_video_path}" -loglevel error

print(f"✅ 25fps video template ready: {final_video_path}")
print("\n📋 Inputs ready! Proceed to Step 3.")


In [ ]:
#@title 🎬 Step 3: Run MuseTalk Inference
#@markdown Choose MuseTalk version and parameters:

version = "v1.5" #@param ["v1.5", "v1.0"]
bbox_shift = 0 #@param {type:"slider", min:-20, max:20, step:1}
use_float16 = True #@param {type:"boolean"}

import os, yaml

MUSETALK_DIR = '/content/MuseTalk'
%cd {MUSETALK_DIR}

# Write YAML config
config_path = os.path.join(MUSETALK_DIR, 'configs', 'inference', 'colab_run.yaml')
os.makedirs(os.path.dirname(config_path), exist_ok=True)

config_data = {
    'task_0': {
        'video_path': '/content/input_data/input_25fps.mp4',
        'audio_path': '/content/input_data/speech_16k.wav',
        'bbox_shift': bbox_shift
    }
}
with open(config_path, 'w') as f:
    yaml.dump(config_data, f)

# Configure version-specific paths
if version == 'v1.5':
    unet_model = './models/musetalkV15/unet.pth'
    unet_config = './models/musetalkV15/musetalk.json'
    ver_flag = '--version v15'
else:
    unet_model = './models/musetalk/pytorch_model.bin'
    unet_config = './models/musetalk/musetalk.json'
    ver_flag = '--version v1'

fp16_flag = '--use_float16' if use_float16 else ''

print(f"\n🚀 Starting MuseTalk ({version}) avatar generation...")
!MPLBACKEND=Agg PYTHONPATH={MUSETALK_DIR} /content/env/bin/python -m scripts.inference \
    --inference_config {config_path} \
    --unet_model_path {unet_model} \
    --unet_config {unet_config} \
    --vae_type sd-vae \
    --whisper_dir ./models/whisper \
    {ver_flag} \
    {fp16_flag}

print("\n✅ Inference finished! Proceed to Step 4 to preview and download.")


In [ ]:
#@title 📺 Step 4: Preview & Download Generated Video
#@markdown Automatically finds your output video, renders an inline player, and triggers download.

import os, glob
from IPython.display import HTML, display
from base64 import b64encode
from google.colab import files

MUSETALK_DIR = '/content/MuseTalk'
%cd {MUSETALK_DIR}

# Find latest generated mp4
results = sorted(glob.glob('results/**/*.mp4', recursive=True), key=os.path.getmtime)

if results:
    output_vid = results[-1]
    print(f"✅ Found Result Video: {output_vid}")
    
    # Inline HTML5 display
    mp4_bytes = open(output_vid, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
    display(HTML(f'''
    <div style="text-align: center; margin: 20px 0;">
        <h3 style="color: #4CAF50;">🎉 Your MuseTalk AI Avatar Video</h3>
        <video width=540 controls autoplay loop style="border-radius: 12px; box-shadow: 0 4px 16px rgba(0,0,0,0.3);">
            <source src="{data_url}" type="video/mp4">
            Your browser does not support HTML5 video.
        </video>
    </div>
    '''))
    
    # Auto-download
    print(f"\n⬇️ Downloading {os.path.basename(output_vid)} to your computer...")
    files.download(output_vid)
else:
    print("❌ No generated video found in results/. Please check Step 3 logs for any error.")


In [ ]:
#@title 🌐 (Optional) Step 5: Launch Gradio Web Interface
#@markdown Launches the interactive MuseTalk Gradio web application with a public shareable URL.

import os
MUSETALK_DIR = '/content/MuseTalk'
%cd {MUSETALK_DIR}

print("🌐 Launching Gradio Web UI...")
!MPLBACKEND=Agg PYTHONPATH={MUSETALK_DIR} /content/env/bin/python app.py --use_float16 --share
